# core

> Fill in a module description here

In [1]:
# | default_exp core

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [3]:
# | export
from typing import Callable
import unittest
import pdb


def run_test_case_with_pdb(
    test_case_class: type[unittest.TestCase], pdb_flag: bool = True
):
    """Run a unittest test case with pdb enabled"""

    class PdbTestResult(unittest.TestResult):
        def addError(self, test: unittest.TestCase, err):
            super().addError(test, err)
            _, _, tb = err
            pdb.post_mortem(tb)

        def addFailure(self, test: unittest.TestCase, err):
            super().addFailure(test, err)
            _, _, tb = err
            pdb.post_mortem(tb)

    suite = unittest.TestSuite()
    suite.addTest(unittest.TestLoader().loadTestsFromTestCase(test_case_class))
    runner = unittest.TextTestRunner(resultclass=PdbTestResult, verbosity=2)
    if not pdb_flag:
        runner = unittest.TextTestRunner(verbosity=2)
        runner.run(suite)
    else:
        runner.run(suite)

In [4]:
# | export
from abc import ABC, abstractmethod
from typing import Any

class AbstractQueue(ABC):
    @abstractmethod
    def enqueue(self, item: Any) -> None:
        """Add an item to the queue."""
        raise NotImplementedError

    @abstractmethod
    def dequeue(self) -> Any:
        """Remove and return an item from the queue."""
        raise NotImplementedError


    @abstractmethod
    def is_empty(self) -> bool:
        """Check if the queue is empty."""
        raise NotImplementedError
    
    @abstractmethod
    def start_workers(self, num_workers: int) -> None:
        """Start worker threads that process jobs from the queue."""
        raise NotImplementedError

    @abstractmethod
    def stop_workers(self) -> None:
        """Stop all worker threads."""
        raise NotImplementedError


In [5]:
import queue
import threading
from typing import Any

class InMemoryQueue(AbstractQueue):
    def __init__(self, maxsize: int = 0):
        self._queue: queue.Queue[Any] = queue.Queue(maxsize=maxsize)
        self._workers: list[threading.Thread] = []

    def enqueue(self, item: Any) -> None:
        self._queue.put(item)

    def dequeue(self) -> Any:
        return self._queue.get()

    def is_empty(self) -> bool:
        return self._queue.empty()

    def _worker(self):
        while True:
            job = self.dequeue()
            if job is None:  # Poison pill means shutdown
                break
            job()

    def start_workers(self, num_workers: int) -> None:
        for _ in range(num_workers):
            thread = threading.Thread(target=self._worker, daemon=True)
            thread.start()
            self._workers.append(thread)

    def stop_workers(self) -> None:
        for _ in self._workers:
            self.enqueue(None)  # Send poison pill to each worker
        for worker in self._workers:
            worker.join()
        self._workers.clear()

In [12]:
import time

class TestInMemoryQueue(unittest.TestCase):
    def test_enqueue_dequeue(self):
        """Test enqueuing and dequeuing items."""
        queue = InMemoryQueue()
        queue.enqueue("item1")
        queue.enqueue("item2")
        self.assertEqual(queue.dequeue(), "item1")
        self.assertEqual(queue.dequeue(), "item2")

    def test_is_empty(self):
        """Test if the queue correctly identifies as empty."""
        queue = InMemoryQueue()
        self.assertTrue(queue.is_empty())
        queue.enqueue("item")
        self.assertFalse(queue.is_empty())
        queue.dequeue()
        self.assertTrue(queue.is_empty())

    def test_worker_processing(self):
        """Test if workers process jobs."""
        results = []
        def job():
            results.append("job done")

        queue = InMemoryQueue()
        queue.start_workers(1)
        queue.enqueue(job)
        time.sleep(0.1)  # Give some time for the job to be processed
        queue.stop_workers()

        self.assertIn("job done", results)

    def test_stop_workers(self):
        """Test if stopping workers clears all workers."""
        queue = InMemoryQueue()
        queue.start_workers(2)
        self.assertEqual(len(queue._workers), 2)
        queue.stop_workers()
        self.assertEqual(len(queue._workers), 0)

suite = unittest.TestSuite()
suite.addTest(unittest.TestLoader().loadTestsFromTestCase(TestInMemoryQueue))
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

test_enqueue_dequeue (__main__.TestInMemoryQueue.test_enqueue_dequeue)
Test enqueuing and dequeuing items. ... ok
test_is_empty (__main__.TestInMemoryQueue.test_is_empty)
Test if the queue correctly identifies as empty. ... ok
test_stop_workers (__main__.TestInMemoryQueue.test_stop_workers)
Test if stopping workers clears all workers. ... ok
test_worker_processing (__main__.TestInMemoryQueue.test_worker_processing)
Test if workers process jobs. ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.106s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

In [7]:
# | export
def foo():
    pass

In [8]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore